In [1]:
import duckdb
import polars as pl
import yfinance as yf
import requests

In [2]:
class DataIngestionEngine:
    def __init__(self, db_path: str = "finagent.duckdb"):
        self.conn = duckdb.connect(db_path)
        self._init_tables()

    def _init_tables(self):
        """Initialize DuckDB analytical schema."""
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS market_prices (
                Ticker VARCHAR,
                Date TIMESTAMP,
                Open DOUBLE,
                High DOUBLE,
                Low DOUBLE,
                Close DOUBLE,
                Volume BIGINT,
                Volatility_20D DOUBLE
            );
        """)

    def fetch_and_store_market_data(self, ticker: str, period: str = "6mo"):
        """Fetch historical stock data, perform rolling metrics transformation via Polars, store in DuckDB."""
        stock = yf.Ticker(ticker)
        df_raw = stock.history(period=period).reset_index()

        if df_raw.empty:
            raise ValueError(f"No market data retrieved for ticker {ticker}")

        # Vectorized feature engineering in Polars
        pl_df = pl.from_pandas(df_raw)
        pl_df = pl_df.with_columns([
            pl.col("Date").cast(pl.Datetime),
            (pl.col("Close").log().diff().rolling_std(window_size=20) * (252 ** 0.5)).alias("Volatility_20D")
        ]).drop_nulls()

        pandas_df = pl_df.to_pandas()
        pandas_df["Ticker"] = ticker.upper()

        # Write to DuckDB
        self.conn.execute("DELETE FROM market_prices WHERE Ticker = ?", [ticker.upper()])
        self.conn.execute("INSERT INTO market_prices SELECT Ticker, Date, Open, High, Low, Close, Volume, Volatility_20D FROM pandas_df")
        print(f"✅ Ingested {len(pandas_df)} market records for {ticker.upper()} into DuckDB.")

    def fetch_sec_company_facts(self, cik: str) -> dict:
        """Query official SEC EDGAR XBRL company facts endpoint."""
        formatted_cik = cik.zfill(10)
        url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{formatted_cik}.json"
        
        # SEC User-Agent requirement header format
        headers = {"User-Agent": "FinAgentX-Platform dev@ubuntu-local.org"}
        
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            print(f"✅ SEC EDGAR facts successfully pulled for CIK {cik}.")
            return response.json()
        else:
            print(f"❌ SEC EDGAR fetch failed. Status code: {response.status_code}")
            return {}

In [3]:
if __name__ == "__main__":
    engine = DataIngestionEngine()
    engine.fetch_and_store_market_data("AAPL")
    engine.fetch_sec_company_facts("0000320193")

✅ Ingested 107 market records for AAPL into DuckDB.
✅ SEC EDGAR facts successfully pulled for CIK 0000320193.


In [4]:
conn = duckdb.connect("finagent.duckdb")

In [5]:
count = conn.execute("SELECT COUNT(*) FROM market_prices WHERE Ticker = 'AAPL'").fetchone()[0]
print(f"count: {count}")

count: 107


In [6]:
df_sample = conn.execute("SELECT * FROM market_prices LIMIT 5").df()
print(df_sample)

  Ticker                Date        Open        High         Low       Close  \
0   AAPL 2026-04-24 04:00:00  272.274102  272.573555  269.169626  270.577118   
1   AAPL 2026-04-27 04:00:00  265.615989  267.881934  264.597817  267.133270   
2   AAPL 2026-04-28 04:00:00  271.854854  272.743283  268.181417  270.227753   
3   AAPL 2026-04-29 04:00:00  267.073362  270.557166  266.564292  269.688721   
4   AAPL 2026-04-30 04:00:00  270.018146  275.508349  267.662365  270.866638   

     Volume  Volatility_20D  
0  38157100        0.262213  
1  41466800        0.258991  
2  40018900        0.256095  
3  30047900        0.240700  
4  91848200        0.240268  
